# Linux basics for scientific computing

The command line is fundamental to scientific computing. While graphical interfaces are convenient for simple tasks, the shell gives you the power to automate workflows, process large datasets, and work efficiently on remote computing clusters. This tutorial covers practical Linux skills you'll use daily in research.

In this tutorial, we will cover:

* The shell environment and navigation
* File operations and viewing content
* Text processing and searching
* Input/output redirection
* File permissions and environment variables
* Process management
* Shell scripting basics
* Remote computing with SSH

## The shell environment

### What is a shell?

A shell is a command-line interpreter that provides an interface to the operating system. When you type commands, the shell interprets them and executes the corresponding programs. The most common shells are:

- **bash** (Bourne Again SHell): The default on most Linux systems
- **zsh**: The default on macOS (since Catalina), with additional features like better tab completion

You can check which shell you're using:

In [ ]:
echo $SHELL

The terms "shell," "terminal," and "command line" are often used interchangeably, though technically the terminal is the application that hosts the shell.

### Navigating the filesystem

The filesystem is organized as a tree structure starting from the root directory `/`. Here are the essential navigation commands:

In [ ]:
pwd                  # Print working directory (where am I?)
ls                   # List files in current directory
ls -l                # Long format with permissions, size, dates
ls -la               # Include hidden files (starting with .)
ls -lh               # Human-readable file sizes
ls -lt               # Sort by modification time
cd /path/to/dir      # Change to specified directory
cd ~                 # Go to home directory
cd ..                # Go up one directory
cd -                 # Go to previous directory

Special directory symbols:
- `.` - Current directory
- `..` - Parent directory
- `~` - Home directory
- `/` - Root directory

Tab completion is one of the most useful shell features. Start typing a path or command and press Tab to autocomplete. Press Tab twice to see all possible completions.

### Getting help

Every command has documentation. There are several ways to access it:

In [ ]:
man ls               # Manual page for ls (press q to exit)
ls --help            # Brief help message
which python         # Show path to executable
type cd              # Show if command is built-in or external

In `man` pages, use:
- Space or `f` to page forward
- `b` to page backward
- `/pattern` to search
- `q` to quit

## File operations

### Creating files and directories

In [ ]:
mkdir data                    # Create a directory
mkdir -p project/data/raw     # Create nested directories
touch analysis.py             # Create empty file (or update timestamp)

### Copying, moving, and deleting

In [ ]:
cp file.txt backup.txt            # Copy file
cp -r dir1 dir2                   # Copy directory recursively
mv old_name.txt new_name.txt      # Rename file
mv file.txt ../                   # Move to parent directory
rm file.txt                       # Delete file (permanent!)
rm -i file.txt                    # Interactive: ask before deleting
rm -r directory                   # Delete directory and contents

**Warning**: `rm` deletes permanently. There is no trash bin. Use `rm -i` when unsure.

### Viewing file contents

Different tools serve different purposes:

In [ ]:
cat file.txt              # Print entire file
head file.txt             # First 10 lines
head -n 20 file.txt       # First 20 lines
tail file.txt             # Last 10 lines
tail -n 5 file.txt        # Last 5 lines
tail -f logfile.log       # Follow file as it grows (useful for logs)
less file.txt             # Pager for large files (use q to exit)
wc file.txt               # Count lines, words, characters
wc -l file.txt            # Count lines only

For large data files, `less` is preferred over `cat` because it loads content on demand rather than dumping everything to the terminal.

### Question

A researcher has a CSV file `patients.csv` with a header row and 500 data rows. They want to see the last 5 data rows (not including the header). Which command would work, and why might `tail -5 patients.csv` not always be the right answer in other scenarios?

### Answer

`tail -5 patients.csv` works here because we just want the last 5 lines. However, if you wanted the last 5 *data* rows and weren't sure if the header was at the top or if there were trailing empty lines, you'd need to be more careful. For example, `tail -n +2 patients.csv | tail -5` skips the header first (shows lines 2 onward), then takes the last 5. The `-n +N` syntax means "start from line N" rather than "show last N lines."

## Text processing and searching

### Searching with grep

`grep` searches for patterns in files. It's one of the most useful commands for working with data and code:

In [ ]:
grep "error" logfile.txt              # Find lines containing "error"
grep -i "error" logfile.txt           # Case-insensitive search
grep -n "error" logfile.txt           # Show line numbers
grep -r "TODO" src/                   # Recursive search in directory
grep -v "^#" config.txt               # Invert: lines NOT matching (exclude comments)
grep -c "patient" data.csv            # Count matching lines
grep -l "import pandas" *.py          # List files containing match

Basic regex patterns:
- `^` - Start of line
- `$` - End of line
- `.` - Any single character
- `*` - Zero or more of preceding character
- `[abc]` - Any character in brackets

In [ ]:
grep "^chr" genes.txt                 # Lines starting with "chr"
grep "error$" log.txt                 # Lines ending with "error"
grep "patient[0-9]" data.txt          # "patient" followed by a digit

### Sorting and counting

In [ ]:
sort file.txt                         # Sort lines alphabetically
sort -n numbers.txt                   # Numeric sort
sort -r file.txt                      # Reverse order
sort -t',' -k2 data.csv               # Sort by 2nd column (comma delimiter)
sort -t',' -k3 -n data.csv            # Sort by 3rd column numerically
uniq file.txt                         # Remove adjacent duplicates
sort file.txt | uniq                  # Remove all duplicates
sort file.txt | uniq -c               # Count occurrences

### Extracting columns

In [ ]:
cut -d',' -f1 data.csv                # Extract 1st column (comma delimiter)
cut -d',' -f1,3 data.csv              # Extract 1st and 3rd columns
cut -d',' -f2-4 data.csv              # Extract columns 2 through 4

### Text substitution with sed

`sed` is a stream editor for transforming text:

In [ ]:
sed 's/old/new/' file.txt             # Replace first occurrence per line
sed 's/old/new/g' file.txt            # Replace all occurrences
sed 's/,/\t/g' file.csv               # Convert CSV to TSV
sed '1d' file.txt                     # Delete first line (header)
sed -n '10,20p' file.txt              # Print lines 10-20

### Combining commands with pipes

The pipe operator `|` sends output of one command as input to another. This is where the command line becomes powerful:

In [ ]:
# Count unique values in column 2 of a CSV
cut -d',' -f2 data.csv | sort | uniq -c | sort -rn

# Find Python files containing "numpy" and count them
grep -l "numpy" *.py | wc -l

# Show top 10 most common words in a file
cat book.txt | tr ' ' '\n' | sort | uniq -c | sort -rn | head -10

# Find running Python processes
ps aux | grep python | grep -v grep

### Question

Given a CSV file `genes.csv` with columns: gene_id, chromosome, expression_level, p_value (with a header row), how would you find the 5 genes with the highest expression levels?

### Answer

In [ ]:
tail -n +2 genes.csv | sort -t',' -k3 -rn | head -5

This skips the header (`tail -n +2`), sorts by the 3rd column (expression_level) in reverse numeric order (`-k3 -rn`), and shows the top 5. If you also wanted just the gene IDs:

In [ ]:
tail -n +2 genes.csv | sort -t',' -k3 -rn | head -5 | cut -d',' -f1

## Input/output redirection

### Standard streams

Every process has three standard streams:
- **stdin** (0): Standard input (keyboard by default)
- **stdout** (1): Standard output (terminal by default)
- **stderr** (2): Standard error (terminal by default)

Understanding these streams is essential for automation.

### Redirecting output

In [ ]:
echo "Hello" > file.txt               # Write to file (overwrites)
echo "World" >> file.txt              # Append to file
ls nonexistent 2> errors.txt          # Redirect stderr to file
python script.py > output.txt 2>&1    # Redirect both stdout and stderr
python script.py &> all.txt           # Shorthand for above (bash)
python script.py > /dev/null 2>&1     # Discard all output

The `>` operator overwrites; `>>` appends. Be careful with `>` as it will erase existing content.

### Redirecting input

In [ ]:
python script.py < input.txt          # Read stdin from file
sort < unsorted.txt > sorted.txt      # Input from file, output to file

### Here documents

Create multi-line input inline:

In [ ]:
cat << EOF > config.txt
database=mydb
host=localhost
port=5432
EOF

### Question

A researcher runs `python simulation.py > results.txt` and later finds `results.txt` is empty, even though they saw output when running the script directly. What likely happened, and how would they capture all output?

### Answer

The script likely wrote to stderr instead of stdout. This happens when using `logging` module with default settings, `sys.stderr.write()`, or print statements with `file=sys.stderr`. The `>` operator only captures stdout.

To capture both:

In [ ]:
python simulation.py > results.txt 2>&1

or

In [ ]:
python simulation.py &> results.txt

To keep them separate:

In [ ]:
python simulation.py > results.txt 2> errors.txt

## File permissions and environment

### Understanding permissions

The `ls -l` output shows permissions:

```
-rw-r--r--  1 user  group  1234 Jan 15 10:30 file.txt
drwxr-xr-x  2 user  group  4096 Jan 15 10:30 directory
```

The permission string has 10 characters:
- Position 1: `d` for directory, `-` for file, `l` for link
- Positions 2-4: Owner permissions (read, write, execute)
- Positions 5-7: Group permissions
- Positions 8-10: Others permissions

### Modifying permissions

In [ ]:
chmod +x script.sh                    # Add execute permission for all
chmod u+x script.sh                   # Add execute for owner only
chmod go-w file.txt                   # Remove write for group and others
chmod 755 script.sh                   # rwxr-xr-x (common for scripts)
chmod 644 file.txt                    # rw-r--r-- (common for files)

Numeric mode uses octal: r=4, w=2, x=1. Sum them for each category (owner, group, others).

### Environment variables

Environment variables configure your shell session and programs:

In [ ]:
echo $PATH                            # Show executable search path
echo $HOME                            # Home directory
echo $USER                            # Current username
echo $PWD                             # Current directory
printenv                              # Show all environment variables

Setting variables:

In [ ]:
MY_VAR="hello"                        # Set shell variable
export MY_VAR="hello"                 # Set and export to child processes
export PATH="$HOME/bin:$PATH"         # Add directory to PATH

Variables set this way only last for the current session. To make them permanent, add them to `~/.bashrc` or `~/.bash_profile`.

### Question

A researcher creates a script `analyze.sh` in their current directory. When they type `analyze.sh`, they get "command not found," but `ls analyze.sh` confirms the file exists. What are two possible causes?

### Answer

1. **The current directory is not in `$PATH`**: The shell only looks for commands in directories listed in `$PATH`. Solution: Run with `./analyze.sh` (explicit path) or add `.` to PATH (not recommended for security reasons).

2. **The file lacks execute permission**: Even if the shell finds it, it can't run without execute permission. Solution: `chmod +x analyze.sh`.

You can check with:

In [ ]:
ls -l analyze.sh    # Check permissions
echo $PATH          # Check if . is in PATH

## Process management

### Viewing processes

In [ ]:
ps                                    # Your processes in current terminal
ps aux                                # All processes, detailed
ps aux | grep python                  # Find Python processes
top                                   # Interactive process viewer (q to quit)
htop                                  # Better interactive viewer (if installed)

The `ps aux` columns: USER, PID, %CPU, %MEM, VSZ, RSS, TTY, STAT, START, TIME, COMMAND

### Running processes in background

In [ ]:
python long_job.py &                  # Run in background
jobs                                  # List background jobs
fg %1                                 # Bring job 1 to foreground
bg %1                                 # Resume stopped job in background

Keyboard shortcuts:
- `Ctrl+C`: Interrupt (kill) current process
- `Ctrl+Z`: Suspend current process (can resume with `fg` or `bg`)

### Terminating processes

In [ ]:
kill 12345                            # Send TERM signal to PID 12345
kill -9 12345                         # Force kill (SIGKILL)
kill %1                               # Kill job number 1
pkill python                          # Kill processes by name

### Persistent processes with nohup

When you log out, background processes normally terminate. Use `nohup` to keep them running:

In [ ]:
nohup python long_simulation.py > output.log 2>&1 &

This runs the script, redirects all output to `output.log`, and continues even after you disconnect.

### Resource monitoring

In [ ]:
time python script.py                 # Measure execution time
free -h                               # Memory usage (Linux)
df -h                                 # Disk space
du -sh data/                          # Size of directory
du -sh * | sort -h                    # Size of items, sorted

## Shell scripting basics

### Creating a script

Shell scripts automate sequences of commands. Create a file with a `.sh` extension:

In [ ]:
#!/bin/bash
# My first script - preprocess_data.sh

echo "Starting data preprocessing..."
python clean_data.py
python normalize.py
python split_train_test.py
echo "Preprocessing complete!"

The first line (`#!/bin/bash`) is the "shebang" - it tells the system which interpreter to use.

Make it executable and run:

In [ ]:
chmod +x preprocess_data.sh
./preprocess_data.sh

### Variables

In [ ]:
#!/bin/bash

# Defining variables (no spaces around =)
NAME="experiment1"
DATE=$(date +%Y%m%d)
OUTPUT_DIR="results/${NAME}_${DATE}"

echo "Creating output directory: $OUTPUT_DIR"
mkdir -p "$OUTPUT_DIR"

Always quote variables that might contain spaces: `"$VARIABLE"` not `$VARIABLE`.

### Command-line arguments

Scripts can accept arguments:

In [ ]:
#!/bin/bash
# process_file.sh - Process a single data file

INPUT_FILE=$1        # First argument
OUTPUT_DIR=$2        # Second argument
THREADS=${3:-4}      # Third argument with default value of 4

echo "Processing $INPUT_FILE"
echo "Output to $OUTPUT_DIR"
echo "Using $THREADS threads"

# $@ is all arguments, $# is the count
echo "Total arguments: $#"

Run with: `./process_file.sh data.csv results/ 8`

### Conditionals

In [ ]:
#!/bin/bash

FILE=$1

if [ -f "$FILE" ]; then
    echo "$FILE exists and is a regular file"
elif [ -d "$FILE" ]; then
    echo "$FILE is a directory"
else
    echo "$FILE does not exist"
fi

Common test conditions:
- `-f file`: File exists and is regular file
- `-d dir`: Directory exists
- `-e path`: Path exists
- `-z string`: String is empty
- `-n string`: String is not empty
- `str1 = str2`: Strings are equal
- `num1 -eq num2`: Numbers are equal
- `num1 -lt num2`: Less than

### Loops

Process multiple files:

In [ ]:
#!/bin/bash
# process_all.sh - Process all CSV files in a directory

INPUT_DIR=$1
OUTPUT_DIR=$2

mkdir -p "$OUTPUT_DIR"

for file in "$INPUT_DIR"/*.csv; do
    if [ -f "$file" ]; then
        filename=$(basename "$file")
        echo "Processing $filename"
        python analyze.py "$file" > "$OUTPUT_DIR/${filename%.csv}_results.txt"
    fi
done

echo "Processed all files"

### Question

A script processes files with a loop, but fails on a file named "patient data.csv" (with a space). The problematic lines are:

In [ ]:
FILES=$(ls *.csv)
for f in $FILES; do
    python process.py $f
done

What is wrong and how should it be fixed?

### Answer

Two problems:
1. Parsing `ls` output is fragile - filenames with spaces become multiple items
2. Variables `$FILES` and `$f` are unquoted, so spaces cause word splitting

Fixed version:

In [ ]:
for f in *.csv; do
    python process.py "$f"
done

The glob pattern `*.csv` handles spaces correctly, and quoting `"$f"` preserves the full filename. Never parse `ls` output; use globs directly.

## Remote computing with SSH

### Connecting to remote servers

SSH (Secure Shell) provides encrypted access to remote systems:

In [ ]:
ssh username@server.unc.edu           # Connect to remote server
ssh -p 2222 user@server.com           # Connect on non-standard port

After connecting, you're in a shell on the remote machine. Type `exit` or `Ctrl+D` to disconnect.

### SSH configuration

Create `~/.ssh/config` to simplify connections:

```
Host cluster
    HostName longleaf.unc.edu
    User yourusername

Host lab
    HostName 192.168.1.100
    User researcher
    Port 2222
```

Now connect with just `ssh cluster` instead of the full command.

### Transferring files

In [ ]:
# Copy local file to remote
scp data.csv user@server:/path/to/destination/

# Copy remote file to local
scp user@server:/path/to/file.txt ./local_copy.txt

# Copy directory recursively
scp -r local_dir/ user@server:/remote/path/

For large transfers or unreliable connections, `rsync` is better:

In [ ]:
# Sync local to remote (only transfers changes)
rsync -avz data/ user@server:/backup/data/

# Resume interrupted transfer
rsync -avz --partial large_file.tar user@server:/destination/

# Sync with progress indicator
rsync -avz --progress results/ user@server:/backup/

`rsync` flags:
- `-a`: Archive mode (preserves permissions, timestamps)
- `-v`: Verbose
- `-z`: Compress during transfer
- `--partial`: Keep partially transferred files for resuming

### Question

A researcher needs to copy 100GB of simulation results from a remote cluster. Their connection sometimes drops. They've been using `scp -r results/ local_results/` but have to restart from scratch each time. What command would be better?

### Answer

In [ ]:
rsync -avz --partial --progress user@cluster:results/ local_results/

`rsync` only transfers files that have changed or are missing. The `--partial` flag keeps incomplete files so transfers can resume. If interrupted, just run the same command again - it will continue where it left off. The `-z` flag compresses data during transfer, which helps with compressible files. For very large directories, add `--delete` if you want to remove local files that no longer exist on the remote.

## Practical tips

### Useful shortcuts

In [ ]:
# Command history
Ctrl+R                 # Search command history (type to search, Enter to run)
!!                     # Repeat last command
!$                     # Last argument of previous command
!grep                  # Run most recent grep command

# Example: create directory then cd into it
mkdir new_project
cd !$                  # Uses "new_project" from previous command

# Run previous command with sudo
sudo !!

### Finding files

In [ ]:
find . -name "*.py"                   # Find Python files
find . -name "*.csv" -mtime -7        # CSV files modified in last 7 days
find . -size +100M                    # Files larger than 100MB
find . -type d -name "test*"          # Directories starting with "test"

### xargs for building commands

`xargs` converts input into arguments:

In [ ]:
# Delete all .tmp files
find . -name "*.tmp" | xargs rm

# Grep in all Python files
find . -name "*.py" | xargs grep "import pandas"

# Run command on each line with placeholder
cat files.txt | xargs -I {} cp {} backup/

### Quick data exploration

In [ ]:
# Preview CSV structure
head -1 data.csv | tr ',' '\n' | nl    # Show columns with numbers

# Count rows (excluding header)
tail -n +2 data.csv | wc -l

# Check for missing values (empty fields)
grep ',,' data.csv | head

# Unique values in a column
cut -d',' -f3 data.csv | sort | uniq

# Quick statistics with awk
awk -F',' 'NR>1 {sum+=$3; count++} END {print "Mean:", sum/count}' data.csv

## Summary

The commands covered in this tutorial form the foundation of working effectively in Linux. Key takeaways:

1. **Navigation**: `pwd`, `ls`, `cd` - know where you are and what's there
2. **File operations**: `cp`, `mv`, `rm`, `mkdir` - manage files carefully
3. **Viewing content**: `cat`, `head`, `tail`, `less`, `wc` - inspect data appropriately
4. **Text processing**: `grep`, `sort`, `uniq`, `cut`, `sed` - transform and search text
5. **Piping**: `|`, `>`, `>>`, `2>&1` - connect commands and redirect output
6. **Permissions**: `chmod` - control access to files
7. **Processes**: `ps`, `kill`, `&`, `nohup` - manage running programs
8. **Scripting**: Variables, loops, conditionals - automate workflows
9. **Remote access**: `ssh`, `scp`, `rsync` - work on remote systems

The command line becomes powerful when you combine simple tools to build complex pipelines. Practice these commands regularly, and you'll find yourself more productive in your research computing work.

### Further resources

- [The Linux Command Line](https://linuxcommand.org/) - comprehensive free book
- [Software Carpentry Shell Tutorial](https://swcarpentry.github.io/shell-novice/) - research-focused
- `man` pages - always available offline
- `tldr` command - simplified man pages (install with package manager)